# Lecture 19 — groupby, and a Toolkit for Messy Files
### EES 3350/5350 · Python in Earth Science · Monday, October 19, 2026 (start of lab)

Earlier, we added columns and asked questions with masks. Lecture 18 ended with a cliffhanger: the average high for **one** month took one line, but all **twelve** months would take twleve lines or a loop.

Today:
1. **`groupby`**: a summary for every month (or every hour, or every day) in one line. This is the main event.
2. **A toolkit for messy files**: five tools you'll need for Lab 08, each with a short exercise.

Then it's lab time.

**Learning objectives**
- use `groupby` to split a table into groups, summarize each one, and get the results back as a Series.
- group by parts of a date index (`.month`, `.hour`, `.date`) and by a column
- find **which** group is biggest or smallest with `.idxmax()` / `.idxmin()` on a groupby result
- recognize `na_values=`, `.value_counts()`, `pd.to_datetime(..., format=...)`, mask assignment with `.loc`, and `.pdTimedelta` when you see them in lab

---
## Part 0 : set up

In [ ]:
import numpy as np
import pandas as pd

raw = pd.read_csv("../../course-materials/week08/nashville_ghcnd_2025.csv", parse_dates=["DATE"], index_col="DATE")
weather = raw[["TMAX", "TMIN", "PRCP", "AWND", "WT03"]].copy()

weather.head()

It's the same Nashville 2025 file as Lectures 17 and 18. `WT03` is the **thunder** column from Wednesday's bonus: `1` on days with thunder and `NaN` on days without.

---
## Part 1: groupby
### The problem
Here's Lecture 18's July average high, and what it would take to get every month:

In [ ]:
for m in range(1, 4):
    print(m, round(weather[weather.index.month == m]["TMAX"].mean(), 1))

That works, but it's a loop that builds a mask, filters, and summarizes, over and over. This "**split** the rows into groups, **apply** a summary to each group, **combine** the answers" pattern is so common that pandas does it one line:

In [ ]:
monthly_high = weather.groupby(weather.index.month)["TMAX"].mean()

monthly_high.round(1)

#### What just happened?
Read `weather.groupby(weather.index.month)["TMAX"].mean()` left to right:
|**piece**|**what it does**|
|:--|:--|
|`wx.groupby(wx.index.month)`|**split**: put every row into a group by its month number (365 rows → 12 groups)|
|`["TMAX"]`|pick the column to summarize|
|`.mean()`|**apply** the mean to each group separately, then **combine** the 12 answers|

```
  rows of wx                     groups                    result
  2025-01-01  TMAX 44  ─┐
  2025-01-02  TMAX 49   ├──▶  month 1: 31 highs  ──mean──▶  1    45.8
  ...                  ─┘
  2025-02-01  TMAX 51  ─┐
  ...                   ├──▶  month 2: 28 highs  ──mean──▶  2    56.1
                       ─┘
  ...                          ...                          ...
  2025-12-31  TMAX 51  ───▶  month 12: 31 highs  ──mean──▶  12   53.2
```
The result is a **Series**: its index is the group labels (the months 1-12), and its **values** are the summaries. July is 94.0, the same as Lecture 15's mask and this morning's one-month version.

Because the result is a Series with the months as its index, one more method draws it. This is a sneak peek: next week is all about making good figures.

In [ ]:
monthly_high.plot(marker="o", xlabel="month", ylabel="average high (F)");

Any summary method works in place of `.mean()`:

In [ ]:
print(weather.groupby(weather.index.month)["PRCP"].sum().round(2))

That's **total** precipitation per month. And because the result is a Series with the months as labels, Wednesday's `.idxmax()` tells you **which** month:

In [ ]:
monthly_rain = weather.groupby(weather.index.month)["PRCP"].sum()

print(f"Wettest month: {monthly_rain.idxmax()} ({monthly_rain.max():.2f} in)")
print(f"Driest month: {monthly_rain.idxmin()} ({monthly_rain.min():.2f} in)")

### Which month had the most thunder?
Wednesday's bonus asked this and promised a faster way (so don't even bother going back). Remember that `.count()` counts the values that **aren't** missing, and `WT03` is `NaN` on days without thunder:

In [ ]:
thunder = weather.groupby(weather.index.month)["WT03"].count()

print(thunder)
print("Most thunder days in month", thunder.idxmax())

So that's 13 thunder days in July. One line isntead of twelve using `.loc` lookups.

### Two summaires, then math

`groupby` results line up by group, so you can do math between them, the same way columns line up by date. The `spread` of daily highs in each month is the biggest high minus the smallest:

In [ ]:
spread = weather.groupby(weather.index.month)["TMAX"].max() - weather.groupby(weather.index.month)["TMAX"].min()

print(spread)

February's highs ranged from 24 $^\circ$F to 80 $^\circ$F: a 56-degree spread in one month. July's stayed within 10 degrees.

### You can group by other things too

The thing inside `groupby(  )` just has to line up with the rows. Some useful ones:

|**group by**|**gives one value per...**|**in Lab 08**|
|:--|:--|:--|
|`wx.index.month`|month (1–12)|Exercise 9|
|`wx.index.hour`|hour of the day (0–23), for data taken more than once a day|Exercise 10|
|`wx.index.date`|calendar day, for data taken more than once a day|Exercise 11|
|`wx.index.dayofweek`|day of the week (0 = Monday)||
|`wx["SOME_COLUMN"]`|each different value in a column||

Our weather file only has one row per day, so `.hour` and `.date` aren't interesting here, but the lab's buoys measure every 3 hours, where they are.

### When a group is empty: `NaN` and `.dropna()`
`WT04` is the sleet column. Gropu it's mean by month:

In [ ]:
sleet = raw.groupby(raw.index.month)["WT04"].mean()

print(sleet)

Months with **no** sleed have nothing to average, so their mean is `NaN`. `.dropna()` removes the `NaN`s from a result and keeps only the groups that had data:

In [ ]:
print(sleet.dropna())

<div class="alert alert-success">

**Tip:** `groupby` follows the same rules as everything else in pandas: `.mean()`, `.sum()`, and `.max()` skip `NaN` inside each group. Use `.count()` to see **how many** values each group's answer is based on. A monthly mean based on 3 measurements deserves less trust than one based on 240.

</div>

<div class="alert alert-info">

### Exercise 1 : one line each

1. What was the **average wind speed** (`AWND`) in each month? Which month was **windiest**, and which was **calmest**?
2. What was the **coldest low** (`TMIN`) in each month?
3. How many days had **any** precipitation in each month? Hint: make a boolean column first, `wx["WET"] = wx["PRCP"] > 0`, then group it and `.sum()` it.
4. **Day of the week:** is it really true that it always rains on the weekend? Find the average precipitation for each day of the week (`wx.index.dayofweek`, 0 = Monday). Which day was wettest in 2025?

</div>

In [ ]:
# your code here
# 1
wind = wx.groupby(___)[___].mean()
print(wind.round(1))
print("Windiest:", wind.___(), " Calmest:", wind.___())

# 2
print(wx.groupby(___)[___].___())

# 3
wx["WET"] = wx["PRCP"] > 0
print(wx.groupby(___)[___].___())

# 4
by_weekday = wx.groupby(___)[___].mean()
print(by_weekday.round(3))

---
## Part 2 : a toolkit for messy files

Lab 08 uses data from ocean buoys, which is messier than our airport file: fill values, quality flags, dates and times in separate text columns, and times in a different time zone. Here are five tools you'll need, one at a time.

### **Tool 1** : `na_values=` turning fill values into `NaN` as you read
Many instruments write a **fill vallue** such as `-999` when they have no reading, like Lecture 15's `-9999`. Our airport file just leaves these cells empty, so to see the problem, let's **make** a file the way a buoy would. `.to_csv()` writes a DataFrame to a file, and a `na_rep=` says what to write in place of `NaN`:

In [ ]:
weather[["TMAX", "AWND"]].to_csv("wind_logger.csv", na_rep=-999)

!head -4 wind_logger.csv
!grep -- -999 wind_logger.csv

Now `wind_logger.csv` is in whatever directory you're working in. It holds the same real data, but the 3 days with no wind reading now say `-999`, just like the bbuoy files (`!grep -- -999` is a terminal command that prints every line containing `-999`). Read it back the normal way:

In [ ]:
logger = pd.read_csv("wind_logger.csv", parse_dates=["DATE"], index_col="DATE")

print(logger["AWND"].min())
print(f"{logger["AWND"].mean():.2f} mph")

A wind speed of -999 mph, and the average is dragged below zero. `na_values=-999` fixed it while teh file is read:

In [ ]:
logger = pd.read_csv("wind_logger.csv", parse_dates=["DATE"], index_col="DATE", na_values=-999)

print(logger["AWND"].min())
print(f"{logger["AWND"].mean():.2f} mph")
print(logger["AWND"].isna().sum(), "missing")

### **Tool 2**: `.value_counts()`, how often each value appears
`.value_counts() lists every different value in a column and how manyh times it appears, most common first:

In [ ]:
(weather["PRCP"] > 0).value_counts()

247 dry days and 118 days with precipitation. In the lab, you'll use it on a **quality flag** column, where each value is a code liek 2 (good) or 4 (bad), to see how much of the data you can trust. The thunder column works like a tiny flag column:

In [ ]:
weather["WT03"].value_counts()

Notice that `.value_counts()` **skips** `NaN`, so only the `1.0`s (61 thunder days) are lists.

### **Tool 3**: writing a mask, `.loc[mask, "col"] = value
In Lecture 18, `.loc[mask, "col"]` **read** values. Put it on the **left** of `=` and it **writes** a value into just those rows; the pandas version of Lecture 15's `a[a == -999] = np.nan`:

In [ ]:
weather["DAY_TYPE"] = "ordinary"
weather.loc[weather["PRCP"] >= 1, "DAY_TYPE"] = "big rain"
weather.loc[weather["TMAX"] >= 95, "DAY_TYPE"] = "scorcher"

weather["DAY_TYPE"].value_counts()

Every row starts as `"ordinary"`, then each mask overwrites only its own rows. In the lab, you'll use the same move to throw out measurements with a bad quality flag.

### **Tool 4**: building dates from text, `pd.to_datetime(..., format=...)`
Our airport file writes dates as `2025-12-25`. If we read ti **without** `parse_dates`, the `DATE` column is just text:

In [ ]:
plain = pd.read_csv("../../course-materials/week08/nashville_ghcnd_2025.csv")

print(plain["DATE"].dtype)
print(plain["DATE"].head(2))

`pd.to_datetime()` turns text into real dates. The `format=` string tells it exactly how the text is written:

In [ ]:
dates = pd.to_datetime(plain["DATE"], format="%Y-%m-%d")

print(dates.dtype)
print(dates.head(2))

|**code**|**means**|**example**|
|:--|:--|:--|
|`%Y`|4-digit year|`2025`|
|`%y`|**2-digit** year|`15`|
|`%m`|month|`12` or `3`|
|`%d`|day|`25`|
|`%H`|hour, 24-hour clock|`18`|
|`%M`|minute|`54`|

The dashes, slashes, spaces, and colons in the format string must match the ones in the text.

**In the lab, the date and time are in two separate text columns**, written like `11/3/15` and `0:17`. You glue them together first. `+` joins strings and on two text **columns** it joins them row by row:

In [ ]:
stamp = pd.to_datetime("11/3/15" + " " + "0:17", format="%m/%d/%y %H:%M")

print(stamp)

### **Tool 5**: shifting times with pd.Timedelta

Scientific instruments almost always record time in **UTC** (Coordinated Universal Time), the same clock everywhere on Earth, so data from different places line up. To get **local** time, add or subtract the difference. `pd.Timedelta` is a length of time that pandas can add to dates. Nashville is 6 hours behind UTC in winter:

In [ ]:
utc = pd.Timestamp("2025-12-25 18:00")
nashville = utc - pd.Timedelta(hours=6)

print(f"UTC:       {utc}")
print(f"Nashville: {nashville}")

Subtracting a `Timedelta` from a whole **index** shifts every timestamp at once: `df.index - df.index - pd.Timedelta(hours=6)`. In the lab this turns out to matter a lot, because the science you'll find happens on **local** time.

## One last trap: `len()` vs. `count()`

- `len(df["col"])` counts **rows**, including the `NaN`s
- `.count()` counts only the values that are **not** missing. When you want "what fraction of the measurements were aove 10 mph," divide by the number of **measurements**:

In [ ]:
windy = (weather["AWND"] > 10).sum()

print(f"windy days:  {windy}")
print(f"rows (len):          {len(weather["AWND"])} -> {windy / len(weather["AWND"]):.1%}")
print(f"measurements (count):{weather["AWND"].count()} -> {windy / weather["AWND"].count():.1%}")


<div class="alert alert-danger">

**Common Error:** dividing by `len()` when a column has missing values. Here only 3 of 365 days are missing, so the difference is tiny. In the lab, one of the columns you'll need is missing in **40–60%** of the rows, and dividing by `len()` gives a badly wrong answer.

</div>

---
## A useful table

|**you want**|**write**|
|:--|:--|
|a summary for every month|`df.groupby(df.index.month)["COL"].mean()`|
|a summary for every value in a column|`df.groupby("COL")["OTHER"].mean()`|
|... for every hour of the day / every day|`df.groupby(df.index.hour)[...]` / `df.groupby(df.index.date)[...]`|
|which group is biggest|`result.idxmax()`|
|drop groups with no data|`result.dropna()`|
|write a DataFrame to a file|`df.to_csv("file.csv", na_rep=-999)`|
|read a file with a fill value|`pd.read_csv(path, na_values=-999)`|
|how often each value appears|`df["COL"].value_counts()`|
|change values only where a mask is True|`df.loc[mask, "COL"] = new_value`|
|text to datetime|`pd.to_datetime(text, format="%m/%d/%y %H:%M")`|
|shift times|`df.index - pd.Timedelta(hours=10)`|
|how many values aren't missing|`df["COL"].count()` (not `len()`)|

**Now: Lab 08.**